In [0]:
import org.apache.spark.sql.cassandra._
//Spark connector
import com.datastax.spark.connector._
import com.datastax.spark.connector.cql.CassandraConnector

//Connection-related
spark.conf.set("spark.cassandra.connection.host","cassandraks.cassandra.cosmos.azure.com")
spark.conf.set("spark.cassandra.connection.port","10350")
spark.conf.set("spark.cassandra.connection.ssl.enabled","true")
spark.conf.set("spark.cassandra.auth.username","cassandraks")
spark.conf.set("spark.cassandra.auth.password","dO0RChKZNa7T6Q903ojd4unPMaL6lFEgPafKfR8zappD6BptKjLAqvEPdinawKAUgG7D0S2dM2eNPdOFeVaikA==")
spark.conf.set("spark.cosmos.diagnostics","simple")


//Read from source table
val sourcetable = sqlContext.read.format("org.apache.spark.sql.cassandra")
  .options(Map("keyspace" -> "datamigration", "table" -> "users"))
  .load

//val cnt = sourcetable.count
//println("count of rows "+cnt)

sourcetable.show(false)

+---------+---------------------+----------+
userid |email |name |
+---------+---------------------+----------+
690675861|97SJcNlOt6@DOMAIN.com|97SJcNlOt6|
693450120|xWZsjO9C6V@DOMAIN.com|xWZsjO9C6V|
703455225|tbDUNwbrFt@DOMAIN.com|tbDUNwbrFt|
690873087|NQSF0piv8Y@DOMAIN.com|NQSF0piv8Y|
693597876|UbpydBBO3Z@DOMAIN.com|UbpydBBO3Z|
693793231|kq4fvvUyuy@DOMAIN.com|kq4fvvUyuy|
695077760|RWVm64SrCV@DOMAIN.com|RWVm64SrCV|
693177035|UgAyJx1czH@DOMAIN.com|UgAyJx1czH|
694948362|xZqwnMfV2a@DOMAIN.com|xZqwnMfV2a|
691535790|ecKyl8Xj3Q@DOMAIN.com|ecKyl8Xj3Q|
692025983|yrsLXqXgBW@DOMAIN.com|yrsLXqXgBW|
695738612|GJs2Y9Uznz@DOMAIN.com|GJs2Y9Uznz|
693115983|B1VysToDOk@DOMAIN.com|B1VysToDOk|
691557612|njqq1y0AqP@DOMAIN.com|njqq1y0AqP|
686145841|DJXhLTZax2@DOMAIN.com|DJXhLTZax2|
684239175|3tzfYtN2Qm@DOMAIN.com|3tzfYtN2Qm|
700866100|pNcth7Ibns@DOMAIN.com|pNcth7Ibns|
691311574|zVliK6K6Te@DOMAIN.com|zVliK6K6Te|
684169048|zy0UgdJpck@DOMAIN.com|zy0UgdJpck|
703689089|69OHVDtfiD@DOMAIN.com|69OHVDtfiD|
+---------+---------------------+----------+
only showing top 20 rows

import org.apache.spark.sql.cassandra._
import com.datastax.spark.connector._
import com.datastax.spark.connector.cql.CassandraConnector
sourcetable: org.apache.spark.sql.DataFrame = [userid: bigint, email: string ... 1 more field]

In [0]:
import java.sql.Timestamp
import org.apache.spark.sql.functions._
import org.apache.spark.sql
import java.nio.charset.StandardCharsets

//var df = sc.parallelize(Seq.range(1, 100)).toDF
var df = sc.parallelize(Seq.range(1, 100000), 10).toDF
//var df = sc.parallelize(Seq.range(1, 1500000)).toDF
//var df = sc.parallelize(Seq.range(1, 2147483640)).toDF
df = df.withColumn("partitionKey",regexp_replace(concat(date_format(expr("reflect('java.time.LocalDateTime', 'now')"),"yyyyMMddhhmmssSSS"), lit(substring(expr("reflect('java.lang.Math', 'random')"),3,2))), "-", ""))
        .withColumn("name", expr("reflect('org.apache.commons.lang3.RandomStringUtils', 'randomAlphanumeric', 10)"))
        .withColumn("email", concat(col("name"), lit("@DOMAIN.com")))
       .withColumn("userid", $"partitionKey".cast(sql.types.LongType))
       .drop("value", "partitionKey")

df.printSchema
df.show(false)

df.write.format("org.apache.spark.sql.cassandra")
  .options(Map("keyspace" -> "datamigration", "table" -> "users"))
  .mode("append")
  .save

root
-- name: string (nullable = true)
-- email: string (nullable = true)
-- userid: long (nullable = true)

+----------+---------------------+-------------------+
name |email |userid |
+----------+---------------------+-------------------+
jumpaAxm70|jumpaAxm70@DOMAIN.com|2022042012043607183|
EzDvRDOyS8|EzDvRDOyS8@DOMAIN.com|2022042012043607264|
ychWphuNbw|ychWphuNbw@DOMAIN.com|2022042012043607287|
PWz1SFaCfr|PWz1SFaCfr@DOMAIN.com|2022042012043607257|
K6oXYY0xKE|K6oXYY0xKE@DOMAIN.com|2022042012043607261|
iEbtTFylKn|iEbtTFylKn@DOMAIN.com|2022042012043607239|
Akv9PcHnN4|Akv9PcHnN4@DOMAIN.com|2022042012043607264|
aowyxE7gk1|aowyxE7gk1@DOMAIN.com|2022042012043607242|
S0urBxViDq|S0urBxViDq@DOMAIN.com|2022042012043607268|
FdnVt9V0m3|FdnVt9V0m3@DOMAIN.com|2022042012043607256|
7L7iK2gMdq|7L7iK2gMdq@DOMAIN.com|2022042012043607226|
Ww6Et16SAm|Ww6Et16SAm@DOMAIN.com|2022042012043607240|
L4XrztlR1H|L4XrztlR1H@DOMAIN.com|2022042012043607238|
neWB66rZZy|neWB66rZZy@DOMAIN.com|2022042012043607265|
duhXTIERf4|duhXTIERf4@DOMAIN.com|2022042012043607217|
cSZgt43dbL|cSZgt43dbL@DOMAIN.com|2022042012043607211|
PzXJ3EbSYX|PzXJ3EbSYX@DOMAIN.com|2022042012043607244|
lrv8kUmVw8|lrv8kUmVw8@DOMAIN.com|2022042012043607273|
V7J7z0Lz8O|V7J7z0Lz8O@DOMAIN.com|2022042012043607261|
agVj8mgyuB|agVj8mgyuB@DOMAIN.com|2022042012043607288|
+----------+---------------------+-------------------+
only showing top 20 rows

import java.sql.Timestamp
import org.apache.spark.sql.functions._
import org.apache.spark.sql
import java.nio.charset.StandardCharsets
df: org.apache.spark.sql.DataFrame = [name: string, email: string ... 1 more field]
df: org.apache.spark.sql.DataFrame = [name: string, email: string ... 1 more field]

In [0]:
//Read from source table
val sourcetable = sqlContext.read.format("org.apache.spark.sql.cassandra")
  .options(Map("keyspace" -> "datamigration", "table" -> "users"))
  .load

sourcetable.write.format("org.apache.spark.sql.cassandra")
  .options(Map("keyspace" -> "datatarget", "table" -> "users"))
  .mode("append")
  .save

sourcetable: org.apache.spark.sql.DataFrame = [userid: bigint, email: string ... 1 more field]

In [0]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql

//Read from source table
val sourcetable = sqlContext.read.format("org.apache.spark.sql.cassandra")
  .options(Map("keyspace" -> "datamigration", "table" -> "users"))
  .load

//Read from target table
val targettable = sqlContext.read.format("org.apache.spark.sql.cassandra")
  .options(Map("keyspace" -> "datatarget", "table" -> "users"))
  .load

val src = sourcetable.withColumn("hashvalue", md5(concat(col("name"), col("email"), col("userid"))))
val tgt = targettable.withColumn("hashvalue", md5(concat(col("name"), col("email"), col("userid"))))

src.repartition(5).createOrReplaceTempView("src3")
tgt.repartition(5).createOrReplaceTempView("tgt3")

sqlContext.sql(
    "SELECT * FROM src3 as s LEFT JOIN tgt3 t on s.userid == t.userid where t.hashvalue is NULL or t.hashvalue != s.hashvalue ").show(false)



+---------+-----------------------+------------+--------------------------------+---------+---------------------+----------+--------------------------------+
userid |email |name |hashvalue |userid |email |name |hashvalue |
+---------+-----------------------+------------+--------------------------------+---------+---------------------+----------+--------------------------------+
54321 |pradip@contoso.com |Pradip |de5ee981b370e82b8ec4a3a5458feb8e|null |null |null |null |
12345 |faiz@contoso.com |Faiz |131e60ff007b6978cdac0ac85bbe82fa|null |null |null |null |
685077821|1RDSnD32Pgad@DOMAIN.com|1RDSnD32Pgad|2353e7f2f250c155d0557ff20883e9ec|685077821|1RDSnD32Pg@DOMAIN.com|1RDSnD32Pg|db1bb4411da02299dddaa005de43dbe0|
+---------+-----------------------+------------+--------------------------------+---------+---------------------+----------+--------------------------------+

import org.apache.spark.sql.functions._
import org.apache.spark.sql
sourcetable: org.apache.spark.sql.DataFrame = [userid: bigint, email: string ... 1 more field]
targettable: org.apache.spark.sql.DataFrame = [userid: bigint, email: string ... 1 more field]
src: org.apache.spark.sql.DataFrame = [userid: bigint, email: string ... 2 more fields]
tgt: org.apache.spark.sql.DataFrame = [userid: bigint, email: string ... 2 more fields]

In [0]:
/*
sqlContext.sql(
    "SELECT * FROM src3 as s where s.userid == 12345").show(false)

sqlContext.sql(
    "SELECT * FROM tgt3 as t where t.userid == 12345").show(false)

sqlContext.sql(
    "SELECT * FROM src3 as s INNER JOIN tgt3 t on s.userid == t.userid where t.userid == 685852761").show(false)

sqlContext.sql(
    "SELECT * FROM src3 as s LEFT JOIN tgt3 t on s.userid == t.userid where s.userid in (12345, 685852761, 54321) and t.hashvalue is NULL ").show(false)
*/

/*
val joined = sqlContext.sql(
    "SELECT * FROM src3 as s LEFT JOIN tgt3 t ON s.userid == t.userid and s.hashvalue != t.hashvalue ")
joined.show(false)
*/
/*
val df2join = tgt.withColumnRenamed("userid", "join_userid")
val joined = src.join(df2join, $"userid" === $"join_userid", "left").drop("join_id")
joined.show(false)
*/